# Wind Interpolation Sweep

Sweep `walks_per_node` for the two sparse GRF variants on the wind interpolation task.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import gpytorch
from tqdm.auto import tqdm
from linear_operator import settings
from gpytorch import settings as gsettings
from grf_gp.sampler import GRFSampler
from grf_gp.utils.spectral import get_normalized_laplacian
from grf_gp.kernels.diffusion import DiffusionGRFKernel
from grf_gp.kernels.general import GeneralGRFKernel
from grf_gp.model import GRFGP
from grf_gp.utils.config import set_gp_defaults

project_root = os.path.abspath("../../..")
sys.path.append(project_root)
sys.path.append(os.path.abspath("."))
from data_utils import prepare_wind_graph_data
from experiments.utils import train_model, evaluate_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_gp_defaults(settings, gsettings)

In [ ]:
CONFIG = {
    "seeds": [0, 1, 2],
    "walks_per_nodes": [32, 64, 128, 256, 512, 1024, 2048, 4096, 8192],
    "models": ["GeneralGRF", "DiffusionGRF"],
    "num_iterations": 1000,
    "learning_rate": 0.01,
    "print_interval": 100,
    "pathwise_samples": 200,
    "downsample_factor": 10,
    "use_downsampling": True,
    "p_halt": 0.1,
    "max_walk_length": 5,
}

NC_FILE = os.path.join(project_root, "data/wind_interpolation/aac1241c2ceff963a460829f1c68b132.nc")
N_PROCESSES = 4

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_data_for_seed(seed):
    set_seed(seed)
    data = prepare_wind_graph_data(
        NC_FILE,
        use_downsampling=CONFIG["use_downsampling"],
        downsample_factor=CONFIG["downsample_factor"],
        dtype=np.float64,
    )

    X_train = torch.tensor(data["X_train"]).long().to(device)
    Y_train = torch.tensor(data["y_train"], dtype=torch.float64).flatten().to(device)
    X_test = torch.tensor(data["X_test"]).long().to(device)
    Y_test = torch.tensor(data["y_test"], dtype=torch.float64).flatten().to(device)

    adjacency_matrix = data["adjacency"].toarray()
    L = get_normalized_laplacian(adjacency_matrix)
    L = torch.tensor(L, dtype=torch.float64).to(device)
    orig_std = float(data["y_std"])
    return X_train, Y_train, X_test, Y_test, L, orig_std


def build_kernel(model_name, rw_mats):
    if model_name == "GeneralGRF":
        return GeneralGRFKernel(rw_mats, CONFIG["max_walk_length"]).to(device)
    if model_name == "DiffusionGRF":
        return DiffusionGRFKernel(rw_mats, CONFIG["max_walk_length"]).to(device)
    raise ValueError(f"Unknown model: {model_name}")


def run_grf(seed, model_name, walks_per_node, rw_mats, X_train, Y_train, X_test, Y_test, orig_std):
    set_seed(seed)
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
    kernel = build_kernel(model_name, rw_mats)
    model = GRFGP(X_train, Y_train, likelihood, kernel).to(device)
    _ = train_model(
        model,
        likelihood,
        X_train,
        Y_train,
        lr=CONFIG["learning_rate"],
        max_iter=CONFIG["num_iterations"],
        print_every=CONFIG["print_interval"],
        progress_desc=f"{model_name} | seed={seed} | walks={walks_per_node}",
    )
    lml, rmse, nlpd = evaluate_model(
        model, likelihood, X_train, Y_train, X_test, Y_test, orig_std
    )
    return {
        "seed": seed,
        "model": model_name,
        "walks_per_node": int(walks_per_node),
        "lml": lml,
        "rmse": rmse,
        "nlpd": nlpd,
    }

In [ ]:
results = []

for seed in tqdm(CONFIG["seeds"], desc="Seeds"):
    X_train, Y_train, X_test, Y_test, L, orig_std = load_data_for_seed(seed)

    for walks_per_node in tqdm(
        CONFIG["walks_per_nodes"],
        leave=False,
        desc=f"Seed {seed} walkers",
    ):
        sampler = GRFSampler(
            L,
            walks_per_node,
            CONFIG["p_halt"],
            CONFIG["max_walk_length"],
            seed=seed,
            use_tqdm=False,
            n_processes=N_PROCESSES,
        )
        print(f"Sampling random walk matrices for seed={seed}, walks_per_node={walks_per_node}")
        rw_mats = sampler()

        for model_name in CONFIG["models"]:
            print(f"Running {model_name} with {walks_per_node} walks/node for seed={seed}")
            results.append(
                run_grf(
                    seed,
                    model_name,
                    walks_per_node,
                    rw_mats,
                    X_train,
                    Y_train,
                    X_test,
                    Y_test,
                    orig_std,
                )
            )

results_df = pd.DataFrame(results)
results_dir = os.path.join(project_root, "experiments/regression/wind_interpolation/results")
os.makedirs(results_dir, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(results_dir, f"wind_interpolation_seed_walk_sweep_{timestamp}.csv")
results_df.to_csv(csv_path, index=False)

print(csv_path)
print(results_df.shape)
results_df.head()